# Predictive Anayltics: Support Vector Machines

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [135]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 1000
SPATIAL_UNIT = "community" # options: census, hexa, community

In [136]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVC 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import load, dump

## Preparations

In [137]:
# "Settings" / Decisions for the training data

# "Settings" / Decisions for the training data
if SPATIAL_UNIT == "census":    
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 
elif SPATIAL_UNIT == "hexa":
    DATA_PATH_TRAIN = "../data/train_test_data/svm_hexa_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_hexa_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_hexa_test.parquet" 
elif SPATIAL_UNIT == "community": 
    DATA_PATH_TRAIN = "../data/train_test_data/svm_community_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_community_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_community_test.parquet" 
else:
    print("Warning: No type of Spatial Unit given, Used census tract")
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 


MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_demand"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_count",
    "date",
]

Load data and select features and target

In [138]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
val = pl.scan_parquet(DATA_PATH_VAL)
test = pl.scan_parquet(DATA_PATH_TEST)

In [139]:
train_df = train.collect()
val_df = val.collect()
test_df = test.collect()

train_df = train_df.to_pandas()
val_df = val_df.to_pandas()
test_df = test_df.to_pandas()

In [140]:
train_df.head()
type(train_df)

pandas.core.frame.DataFrame

In [141]:
# prepare data
# calculate median to split in low/high demand
# when trip_count above 50 percent use "high", when below or equal to 50 percent low


# definition for demand cause currently the q1 is 0, q2 is 1 and q3 is 3
# I excluded the zeros since a magority of values are zero
train_p90 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 90)
train_p70 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 70)
train_p50 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 50)
train_p25 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 25)

train_df["trip_demand"] = "Low"
train_df.loc[train_df["trip_count"] >= train_p25, "trip_demand"] = "Mid"
train_df.loc[train_df["trip_count"] >= train_p50, "trip_demand"] = "Mid High"
train_df.loc[train_df["trip_count"] >= train_p70, "trip_demand"] = "High"
train_df.loc[train_df["trip_count"] >= train_p90, "trip_demand"] = "Very High"

val_p90 = np.percentile(val_df.loc[val_df["trip_count"] > 0, "trip_count"], 90)
val_p70 = np.percentile(val_df.loc[val_df["trip_count"] > 0, "trip_count"], 70)
val_p50 = np.percentile(val_df.loc[val_df["trip_count"] > 0, "trip_count"], 50)
val_p25 = np.percentile(val_df.loc[val_df["trip_count"] > 0, "trip_count"], 25)

val_df["trip_demand"] = "Low"
val_df.loc[val_df["trip_count"] >= val_p25, "trip_demand"] = "Mid"
val_df.loc[val_df["trip_count"] >= val_p50, "trip_demand"] = "Mid High"
val_df.loc[val_df["trip_count"] >= val_p70, "trip_demand"] = "High"
val_df.loc[val_df["trip_count"] >= val_p90, "trip_demand"] = "Very High"

test_p90 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 90)
test_p70 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 70)
test_p50 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 50)
test_p25 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 25)

test_df["trip_demand"] = "Low"
test_df.loc[test_df["trip_count"] >= test_p25, "trip_demand"] = "Mid"
test_df.loc[test_df["trip_count"] >= test_p50, "trip_demand"] = "Mid High"
test_df.loc[test_df["trip_count"] >= test_p70, "trip_demand"] = "High"
test_df.loc[test_df["trip_count"] >= test_p90, "trip_demand"] = "Very High"


In [142]:
print(test_df.loc[test_df["trip_count"] == 0, "trip_count"].count())
print(test_df.loc[ test_df["trip_count"] > 0, "trip_count"].count())

104324
132928


In [143]:
train_p90

np.float64(26.0)

In [144]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

# Community_area is a categorical id, not a numeric quantity, so one-hot encode it
X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])

# Keep the dummy columns before scaling turns X_train into a plain array
train_columns = X_train.columns

# Make sure val/test have the same dummy columns as train (in case a community_area is missing)
X_val = X_val.reindex(columns=train_columns, fill_value=0)
X_test = X_test.reindex(columns=train_columns, fill_value=0)

y_train = train_df[TARGET_COL]
y_val = val_df[TARGET_COL]
y_test = test_df[TARGET_COL]

# SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Features:", feature_cols)
print("Target:", y_train.dtypes)

Features: ['month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'hour_sin', 'hour_cos', 'tmpc', 'relh', 'sknt', 'vsby', 'p01m', 'skyc1_BKN', 'skyc1_CLR', 'skyc1_FEW', 'skyc1_OVC', 'skyc1_SCT', 'skyc1_VV ', 'is_holiday', 'community_area', 'food_drink', 'landmark', 'shop', 'train_station']
Target: object


In [145]:
model = SVC()

In [146]:
train_df = train_df.sample(n=5000, random_state=42)
train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)

In [147]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

# Community_area is a categorical id, not a numeric quantity, so one-hot encode it
X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])

# Keep the dummy columns before scaling turns X_train into a plain array
train_columns = X_train.columns

# Make sure val/test have the same dummy columns as train (in case a community_area is missing)
X_val = X_val.reindex(columns=train_columns, fill_value=0)
X_test = X_test.reindex(columns=train_columns, fill_value=0)

y_train = train_df[TARGET_COL]
y_val = val_df[TARGET_COL]
y_test = test_df[TARGET_COL]

# SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Features:", feature_cols)
print("Target:", y_train.dtypes)

Features: ['month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'hour_sin', 'hour_cos', 'tmpc', 'relh', 'sknt', 'vsby', 'p01m', 'skyc1_BKN', 'skyc1_CLR', 'skyc1_FEW', 'skyc1_OVC', 'skyc1_SCT', 'skyc1_VV ', 'is_holiday', 'community_area', 'food_drink', 'landmark', 'shop', 'train_station']
Target: object


In [148]:
# Create X and y for grid search (same encoding + scaling as the full training set)
X_train_grid = pd.get_dummies(train_df_grid[feature_cols], columns=["community_area"])
X_train_grid = X_train_grid.reindex(columns=train_columns, fill_value=0)
X_train_grid = scaler.transform(X_train_grid)

y_train_grid = train_df_grid[TARGET_COL]

In [149]:
param_grid_linear = {
    "C": [0.1, 1, 10],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [0.1, 1, 10],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "kernel": ["poly"],
    "degree": [3, 4],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVC(),
        param_grid=grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_train_grid, y_train_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.5699981418544293 best params: {'C': 0.1, 'kernel': 'linear'}
rbf_sigmoid best score: 0.5629881378384373 best params: {'C': 10, 'gamma': 'auto', 'kernel': 'sigmoid'}
poly best score: 0.5310010609411807 best params: {'C': 1, 'degree': 3, 'gamma': 'scale', 'kernel': 'poly'}
Overall best: linear {'C': 0.1, 'kernel': 'linear'}


In [150]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 0.1, 'kernel': 'linear'}
Best CV score: 0.5699981418544293


In [151]:
# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

#rint("Test accuracy:", accuracy_score(y_test, y_pred))

In [152]:
# Train SVC 
best_model.fit(X_train, y_train)

,C,0.1
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [153]:
# Make prediction 
y_pred = grid_search.predict(X_test)

In [154]:
# Evaluate Model
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

[[17220   403   317  6315  3359]
 [  779 89185  5408  8890    62]
 [ 1125 28425  3997 12496    75]
 [ 5759 13325  3407 23077   237]
 [ 1808     1     0    59 11523]]
              precision    recall  f1-score   support

        High       0.65      0.62      0.63     27614
         Low       0.68      0.85      0.76    104324
         Mid       0.30      0.09      0.13     46118
    Mid High       0.45      0.50      0.48     45805
   Very High       0.76      0.86      0.80     13391

    accuracy                           0.61    237252
   macro avg       0.57      0.59      0.56    237252
weighted avg       0.56      0.61      0.57    237252



In [155]:
# save model
dump(best_model, "../models/model_" + SPATIAL_UNIT + "_svc.joblib")
dump(grid_search, "../models/grid_" + SPATIAL_UNIT + "_svc.joblib")

['../models/grid_community_svc.joblib']